# DB061 — Verify Final Dataset Record Counts (v2, corrected per Anjum's review)

**What changed from v1:** the first version of this notebook traced `products_5k_enriched.json` → `products_enriched.json`, which is **v1.0's** pipeline (it released 4,731 records). The v1.1 manifest's own `source` field names a completely different lineage — this version traces that real chain instead, with every comparison as an `assert` rather than a printed number you have to eyeball.

**Confirmed v1.1 lineage:** six raw source files (61,373 records) → DB069's quality-based candidate selection (5,000 kept, 56,373 excluded) → the DB069 candidate (5,000 records) → DB070's clean/enrich/validate run → the v1.1 release (5,000 records).

## Setup
All paths below are confirmed against real file reads and the manifest's own stated hashes — not guessed.

In [1]:
import json
import gzip
import hashlib
from pathlib import Path

# Auto-detects the repo root whether this notebook is run from scripts/ or the repo root itself
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "database").exists() and (REPO_ROOT.parent / "database").exists():
    REPO_ROOT = REPO_ROOT.parent
print(f"Repo root resolved to: {REPO_ROOT.resolve()}")

# --- DB069: candidate selection (the real v1.1 input stage) ---
DB069_DIR = REPO_ROOT / "database" / "Candidates" / "DB069"
DB069_GENERATION_RECORD = DB069_DIR / "generation_record.json"
DB069_CANDIDATE = DB069_DIR / "foodremedy_candidate_db069.json"
DB069_INCLUSION = DB069_DIR / "inclusion_ledger.jsonl"
DB069_EXCLUSION_GZ = DB069_DIR / "exclusion_ledger.jsonl.gz"

# --- v1.1 release evidence ---
VALIDATION_MANIFEST = REPO_ROOT / "database" / "Release" / "v1.1" / "validation_manifest.json"
DB060_VALIDATION_V11 = REPO_ROOT / "database" / "Release" / "v1.1" / "db060_validation.json"
FINAL_RELEASE_PATH = REPO_ROOT / "database" / "Release" / "v1.1" / "foodremedy_release_v1.1.json"

# --- Ruled-out path: v1.0's pipeline, kept only to document why it was checked and excluded ---
V1_0_INPUT_PATH = REPO_ROOT / "database" / "seeding" / "products_5k_enriched.json"
V1_0_ENRICHED_PATH = REPO_ROOT / "database" / "seeding" / "products_enriched.json"


Repo root resolved to: C:\Users\GAURAV\OneDrive\Desktop\Food Remedy API - Project\Food-Remedy_API


## Helper functions
`check()` does two things at once: it prints a readable PASS/FAIL line, and it raises an `AssertionError` naming the exact mismatch if the numbers don't agree. That's what makes this notebook's checks reproducible rather than something a reader has to verify by eye.

In [2]:
def check(label, actual, expected):
    status = "PASS" if actual == expected else "FAIL"
    print(f"[{status}] {label}: got {actual!r}, expected {expected!r}")
    assert actual == expected, f"{label}: got {actual!r}, expected {expected!r}"

def count_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return sum(1 for l in f if l.strip())

def count_jsonl_gz(path):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        return sum(1 for l in f if l.strip())

def count_json_array(path):
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    return len(data), data

def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_of_normalized(path):
    with open(path, "rb") as f:
        data = f.read().replace(b"\r\n", b"\n")  # undo Windows-style line breaks
    return hashlib.sha256(data).hexdigest()


## Stage 0 — Raw source pool
DB069's `generation_record.json` names six raw files as its starting pool. Sum their stated record counts — this is the true "input" number for v1.1, not the 5,000 we originally (wrongly) used.

In [3]:
with open(DB069_GENERATION_RECORD, encoding="utf-8") as f:
    generation_record = json.load(f)

source_files = generation_record["source_files"]
raw_input_total = sum(f["records"] for f in source_files)

for f in source_files:
    print(f"  {f['file']}: {f['records']} records")
print(f"\nTotal raw input across {len(source_files)} files: {raw_input_total}")


  database/seeding/products_0k_10k.json: 10000 records
  database/seeding/products_10k_20k.json: 10001 records
  database/seeding/products_20k_30k.json: 10001 records
  database/seeding/products_30k_40k.json: 10002 records
  database/seeding/products_40k_50k.json: 10002 records
  database/seeding/products_50k+.json: 11367 records

Total raw input across 6 files: 61373


## Stage 1 — DB069 selection reconciliation
Every raw record should land in exactly one of two places: the inclusion ledger (kept) or the exclusion ledger (dropped, with a reason). If they don't sum to the raw total, something is unaccounted for.

In [4]:
db069_included = count_jsonl(DB069_INCLUSION)
db069_excluded = count_jsonl_gz(DB069_EXCLUSION_GZ)

print(f"Included: {db069_included}")
print(f"Excluded: {db069_excluded}")
check("DB069 included + excluded == raw input total", db069_included + db069_excluded, raw_input_total)


Included: 5000
Excluded: 56373
[PASS] DB069 included + excluded == raw input total: got 61373, expected 61373


## Stage 2 — DB069 candidate identity
The candidate file itself should match the inclusion count exactly, and its hash should match both `generation_record.json` and the v1.1 manifest's `source.candidate_sha256` — proving the manifest is really pointing at this exact file.

In [5]:
db069_candidate_count, _ = count_json_array(DB069_CANDIDATE)
db069_candidate_hash_raw = sha256_of(DB069_CANDIDATE)
db069_candidate_hash = sha256_of_normalized(DB069_CANDIDATE)

print(f"Raw local hash (Windows checkout): {db069_candidate_hash_raw}")
print(f"Normalized (LF) hash:              {db069_candidate_hash}")

check("DB069 candidate record count == included count", db069_candidate_count, db069_included)
check("DB069 candidate hash (normalized) == generation_record's stated candidate hash",
      db069_candidate_hash, generation_record["candidate"]["sha256"])

with open(VALIDATION_MANIFEST, encoding="utf-8") as f:
    manifest = json.load(f)

check("DB069 candidate hash (normalized) == v1.1 manifest's source.candidate_sha256",
      db069_candidate_hash, manifest["source"]["candidate_sha256"])
check("v1.1 manifest's source.candidate_records == 5000",
      manifest["source"]["candidate_records"], 5000)

Raw local hash (Windows checkout): 7a2104b4ca4a9fee062c4ada151d4c015c1376907484eba5f89cd7da46a449b9
Normalized (LF) hash:              ffc2d26e4cc1fdaa8e1754e209f0be0b7fe7b93784b743b47911180d52b67ced
[PASS] DB069 candidate record count == included count: got 5000, expected 5000
[PASS] DB069 candidate hash (normalized) == generation_record's stated candidate hash: got 'ffc2d26e4cc1fdaa8e1754e209f0be0b7fe7b93784b743b47911180d52b67ced', expected 'ffc2d26e4cc1fdaa8e1754e209f0be0b7fe7b93784b743b47911180d52b67ced'
[PASS] DB069 candidate hash (normalized) == v1.1 manifest's source.candidate_sha256: got 'ffc2d26e4cc1fdaa8e1754e209f0be0b7fe7b93784b743b47911180d52b67ced', expected 'ffc2d26e4cc1fdaa8e1754e209f0be0b7fe7b93784b743b47911180d52b67ced'
[PASS] v1.1 manifest's source.candidate_records == 5000: got 5000, expected 5000


## Stage 3 — DB070's validation of the candidate
DB070 reran DB060's validation as part of producing v1.1. This should show every one of the 5,000 candidate records as valid, with zero invalid.

In [6]:
with open(DB060_VALIDATION_V11, encoding="utf-8") as f:
    db060_v11 = json.load(f)

check("v1.1 total_records == DB069 candidate count", db060_v11["total_records"], db069_candidate_count)
check("v1.1 valid_records == total_records", db060_v11["valid_records"], db060_v11["total_records"])
check("v1.1 invalid_records == 0", db060_v11["invalid_records"], 0)


[PASS] v1.1 total_records == DB069 candidate count: got 5000, expected 5000
[PASS] v1.1 valid_records == total_records: got 5000, expected 5000
[PASS] v1.1 invalid_records == 0: got 0, expected 0


## Stage 4 — Final release identity
The final release file's record count and hash should match what DB070's own README and the manifest both state.

In [7]:
final_count, final_records = count_json_array(FINAL_RELEASE_PATH)
final_raw_hash = sha256_of(FINAL_RELEASE_PATH)
final_normalized_hash = sha256_of_normalized(FINAL_RELEASE_PATH)

barcodes = [r.get("barcode") for r in final_records if r.get("barcode")]
unique_barcodes = len(set(barcodes))

check("Final release record count == 5000", final_count, 5000)
check("Final release record count == DB069 candidate count", final_count, db069_candidate_count)
check("Unique barcodes == record count (no duplicates)", unique_barcodes, final_count)
check("Final release hash (normalized) == manifest's dataset.sha256",
      final_normalized_hash, manifest["dataset"]["sha256"])

# DB070's README states this exact hash as the canonical, twice-reproduced artifact hash
DB070_STATED_HASH = "43b955c6837eda262b5b9afba5e8227ae7a1db3538dab110b393b8e9920791e1"
check("Final release hash (normalized) == DB070 README's stated hash",
      final_normalized_hash, DB070_STATED_HASH)

print(f"\nRaw local hash (Windows checkout): {final_raw_hash}")
print("This differs from the approved hash — see the line-ending note below. It is not a second content difference.")


[PASS] Final release record count == 5000: got 5000, expected 5000
[PASS] Final release record count == DB069 candidate count: got 5000, expected 5000
[PASS] Unique barcodes == record count (no duplicates): got 5000, expected 5000
[PASS] Final release hash (normalized) == manifest's dataset.sha256: got '43b955c6837eda262b5b9afba5e8227ae7a1db3538dab110b393b8e9920791e1', expected '43b955c6837eda262b5b9afba5e8227ae7a1db3538dab110b393b8e9920791e1'
[PASS] Final release hash (normalized) == DB070 README's stated hash: got '43b955c6837eda262b5b9afba5e8227ae7a1db3538dab110b393b8e9920791e1', expected '43b955c6837eda262b5b9afba5e8227ae7a1db3538dab110b393b8e9920791e1'

Raw local hash (Windows checkout): a1635e222b9e5fbec2ca07fee16bbed1f275a168dc98945744ba1d5f49351949
This differs from the approved hash — see the line-ending note below. It is not a second content difference.


## Stage 5 — Manifest's own workflow counts (independent cross-check)
The manifest records DB070's cleaning and enrichment steps directly. Note: `inclusion_count`/`exclusion_count` here means something different from DB069's ledgers above — this is DB070 accepting/rejecting records **out of the already-selected 5,000-record candidate**, not re-selecting from the raw 61,373. Conflating these two would be a mistake.

In [8]:
cleaning = manifest["workflow"]["cleaning"]
enrichment = manifest["workflow"]["enrichment"]

check("DB070 cleaning input_records == DB069 candidate count", cleaning["input_records"], db069_candidate_count)
check("DB070 cleaning output_records == input_records (no loss)", cleaning["output_records"], cleaning["input_records"])
check("DB070 cleaning excluded_records == 0", cleaning["excluded_records"], 0)
check("DB070 enrichment processed == 5000", enrichment["processed"], 5000)
check("DB070 enrichment failures == 0", enrichment["failures"], 0)
check("DB070 release-stage inclusion_count == 5000", manifest["workflow"]["inclusion_count"], 5000)
check("DB070 release-stage exclusion_count == 0", manifest["workflow"]["exclusion_count"], 0)


[PASS] DB070 cleaning input_records == DB069 candidate count: got 5000, expected 5000
[PASS] DB070 cleaning output_records == input_records (no loss): got 5000, expected 5000
[PASS] DB070 cleaning excluded_records == 0: got 0, expected 0
[PASS] DB070 enrichment processed == 5000: got 5000, expected 5000
[PASS] DB070 enrichment failures == 0: got 0, expected 0
[PASS] DB070 release-stage inclusion_count == 5000: got 5000, expected 5000
[PASS] DB070 release-stage exclusion_count == 0: got 0, expected 0


## Ruled-out path — documented for the record
`products_5k_enriched.json` → `products_enriched.json` both independently count to 5,000, which is why this looked like the right chain at first. It is **v1.0's** pipeline, not v1.1's — the manifest's own `source` field and the version-delta below prove they're materially different datasets, not two names for the same one.

In [9]:
v1_0_input_count, _ = count_json_array(V1_0_INPUT_PATH)
v1_0_enriched_count, _ = count_json_array(V1_0_ENRICHED_PATH)
print(f"products_5k_enriched.json: {v1_0_input_count} records (v1.0's input — NOT part of the v1.1 chain)")
print(f"products_enriched.json:    {v1_0_enriched_count} records (v1.0's enriched output — NOT part of the v1.1 chain)")

delta = manifest["delta_summary"]
print(f"\nv1.0 -> v1.1 delta: added={delta['added']}, removed={delta['removed']}, "
      f"updated={delta['updated']}, unchanged={delta['unchanged']}")
print(f"Only {delta['unchanged'] + delta['updated']} of v1.0's {delta['v1.0_records']} records "
      f"carried through to v1.1 in any form — confirms these are different datasets, not the same one renamed.")


products_5k_enriched.json: 5000 records (v1.0's input — NOT part of the v1.1 chain)
products_enriched.json:    5000 records (v1.0's enriched output — NOT part of the v1.1 chain)

v1.0 -> v1.1 delta: added=4878, removed=4609, updated=113, unchanged=9
Only 122 of v1.0's 4731 records carried through to v1.1 in any form — confirms these are different datasets, not the same one renamed.


## Summary — the confirmed v1.1 chain

| Stage | Count | Evidence |
| --- | ---: | --- |
| Raw source pool (6 files) | 61,373 | DB069 `generation_record.json` |
| DB069 excluded | 56,373 | `exclusion_ledger.jsonl.gz` |
| DB069 included (candidate) | 5,000 | `inclusion_ledger.jsonl`, hash-matched to manifest |
| DB070 cleaning/enrichment output | 5,000 | `validation_manifest.json` workflow section |
| DB070 validation (DB060 rerun) | 5,000 valid, 0 invalid | `db060_validation.json` (v1.1) |
| Final release | 5,000, hash-matched (normalized) | `foodremedy_release_v1.1.json`, DB070 README |

Every cell above either passed its assertion or the notebook would have stopped with an error naming the exact mismatch. All of them passed.

## Differences & findings

- **Wrong lineage in the first draft of this verification**, caught in review: the `products_5k_enriched.json` → `products_enriched.json` chain belongs to v1.0, not v1.1. Corrected above; the real v1.1 lineage is traced and hash-verified end to end.
- **Raw file hash does not match the approved hash on a Windows checkout.** Confirmed root cause: Git's `core.autocrlf=true` converts LF to CRLF on checkout. The normalized hash matches exactly, and DB070 independently reproduced the same hash twice during its own build — so there is no content loss. This isn't limited to the final release file either: the DB069 candidate file (`foodremedy_candidate_db069.json`) shows the exact same raw/normalized split (raw `7a2104b4...` vs. normalized `ffc2d26e...`, which matches `generation_record.json` and the manifest exactly). The repo lacks a `.gitattributes` enforcing LF for release artifacts, which is the actual gap and is worth raising separately — it affects any release-stage JSON file checked out on Windows, not just one.
- No other differences found. Every stage from the raw 61,373-record pool down to the 5,000-record final release reconciles exactly.